# P34 — RoFormer: Transformer mejorado con codificación posicional rotatoria

## 1. Título y paper

**Paper:** *RoFormer: Enhanced Transformer with Rotary Position Embedding*  
**Autoría:** Jianlin Su, Yu Lu, Shengfeng Pan, Ahmed Murtadha, Bo Wen, Yunfeng Liu  
**Año y venue:** 2021 · arXiv:2104.09864  
**Nivel:** L3 · **Motor:** `rope`  
**Ficha completa:** [`P34_rope`](../../papers/foundational/P34_rope/README.md)

**Hito:** La posición se codifica rotando, y la atención pasa a depender solo de la distancia relativa. Es la base de casi todo modelo actual.

- [arXiv:2104.09864](https://arxiv.org/abs/2104.09864)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La codificación sinusoidal del Transformer se SUMA al embedding y codifica posición absoluta; la atención no ve directamente la distancia entre dos tokens, que es lo que importa en lenguaje.
2. Ejecutar una implementación mínima de la propuesta: Rotar los vectores de consulta y clave en función de su posición, de modo que el producto escalar entre dos posiciones dependa únicamente de su diferencia.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P08


## 4. Intuición

En vez de sumar una marca de posición, se **rota** el vector según dónde esté. Al comparar dos tokens, las rotaciones se cancelan parcialmente y lo que queda depende solo de cuánto se separan. La posición absoluta desaparece del resultado.


## 5. Concepto mínimo

```text
RoPE: q_m = R_m·q,   k_n = R_n·k,   con R_θ una rotación por bloques de 2

    ⟨R_m·q, R_n·k⟩ = f(q, k, m − n)      ← solo la DIFERENCIA
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('rope', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Darán el mismo producto escalar las posiciones (5,3) y (500,498)?
2. ¿Qué le pasa al producto conforme crece la distancia?
3. ¿Por qué eso es un buen sesgo para lenguaje?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('rope', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('rope', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Las tres parejas con la misma diferencia dan **exactamente** el mismo valor. La posición absoluta se usa para rotar, pero no aparece en el resultado: la atención solo ve distancia relativa.


## 10. Comentario pedagógico

Esto es lo que hoy llevan casi todos los modelos abiertos. La codificación sinusoidal de [P08](../../papers/foundational/P08_transformer/README.md) sigue siendo válida, pero RoPE se impuso porque da la relatividad **gratis**, sin parámetros extra ni tablas de posición.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que RoPE permite por sí solo extrapolar a contextos mucho más largos.


In [ ]:
print('RoPE da posicion relativa; NO garantiza extrapolar mas alla del entrenamiento.')
print('Extender el contexto exige tecnicas POSTERIORES (interpolacion de posiciones).')
print('Atribuirle eso al paper de 2021 es un anacronismo.')

## 12. Corrección

Lo que sí aporta, enunciado con precisión:


In [ ]:
aporta = {'relatividad': 'el producto depende solo de m−n',
          'sin parametros': 'la rotacion no anade pesos que aprender',
          'decaimiento': 'tiende a bajar con la distancia, buen sesgo para lenguaje',
          'no_aporta': 'extrapolacion automatica a longitudes no vistas'}
show(aporta)

## 13. Desafío guiado

Comprueba que dos parejas con distinta diferencia dan valores distintos, y que la de diferencia 0 es la mayor.


In [ ]:
r = run_paper_lab('rope', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa RoPE sobre una atención pequeña y mide la exactitud en una tarea de copia con posiciones desplazadas. Comprueba si el modelo generaliza a posiciones que no vio.


## 15. Evidencia de aprendizaje

Guarda la tabla de invariancia relativa, la de decaimiento y tu enunciado de qué aporta y qué no.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P34_rope/README.md) · evaluación formal: [`assessments/papers/P34_rope.md`](../../assessments/papers/P34_rope.md)


## 16. Cierre

La posición ya es relativa y barata. El siguiente muro no es matemático: es la memoria del hardware.


## 17. Conexión con el siguiente hito

- P35
- extensión de contexto por interpolación

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
